# Multivariate Regression for House Price Prediction from a .csv File

## Workflow:
1. **Load and Inspect Data**: Use `pandas` to load the `data.csv` file and perform essential data quality checks.
2. **Data Visualization**: Use `seaborn` and `matplotlib` to explore relationships between features.
3. **Data Preprocessing**: Split the data into training and testing sets and apply feature scaling.
4. **Model Building**: Define a neural network for regression using PyTorch.
5. **Training**: Train the model and visualize the loss curves.
6. **Evaluation**: Evaluate the model on the test set and visualize the results.
7. **Save, Load, and Infer**: Save the trained model, load it back, and use it to predict the price of a new, unseen house.

## 1. Imports

We'll import `pandas` for data handling, `sklearn` for preprocessing, `matplotlib` and `seaborn` for plotting, and `torch` for the neural network.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

## 2. Load and Inspect the Data

We start by loading our `data.csv` file into a pandas DataFrame.

In [ ]:
df = pd.read_csv('data.csv')
df.head()

Let's check the data types and for any missing values.

In [ ]:
df.info()

Get summary statistics to understand the data distribution.

In [ ]:
df.describe()

## 3. Data Visualization

Visualizing the relationships between features can provide valuable insights.

In [ ]:
sns.pairplot(df) # For visualizing relationships. Pairplot shows scatterplots for each pair of features.
plt.show()

A heatmap of correlations can show which features are most related to the price.

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(), annot=True, cmap='coolwarm') # Correlation heatmap
plt.show()

## 4. Data Preprocessing

In [ ]:
X = df[['SquareFeet', 'NumBedrooms', 'NumBathrooms']].values
y = df['Price'].values.reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
scaler = StandardScaler() # Standardize features by removing the mean and scaling to unit variance
X_train_scaled = scaler.fit_transform(X_train) # Fit to training data and transform
X_test_scaled = scaler.transform(X_test) # Transform test data

y_scaler = StandardScaler() # Standardize target variable
y_train_scaled = y_scaler.fit_transform(y_train)   # y_train is (n,1)
y_test_scaled = y_scaler.transform(y_test)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32) # Train features tensor
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32) # Test features tensor
y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32)

## DataLoader for batching

In [ ]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=10, shuffle=True)

train_dataset.tensors[:10]

In [ ]:
train_loader.batch_size

## 5. Model Building

In [ ]:
class RegressionModel(nn.Module):
    def __init__(self, input_size):
        super(RegressionModel, self).__init__()
        self.layer1 = nn.Linear(input_size, 32)
        self.relu1 = nn.ReLU()
        self.layer2 = nn.Linear(32, 16)
        self.relu2 = nn.ReLU()
        self.output_layer = nn.Linear(16, 1)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu1(x)
        x = self.layer2(x)
        x = self.relu2(x)
        x = self.output_layer(x)
        return x

## 6. Training the Model

In [ ]:
input_size = X_train_tensor.shape[1]

model = RegressionModel(input_size)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

num_epochs = 100
train_losses = []

for epoch in range(num_epochs):
    # model.train() # Set model to training mode
    
    epoch_loss = 0.0
    for inputs, targets in train_loader: # Mini-batch training
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    train_losses.append(epoch_loss / len(train_loader))
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss / len(train_loader):.4f}')

In [ ]:
train_losses

### Visualize Training Loss

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(train_losses, label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training Loss Over Time')
plt.legend()
plt.show()

## 7. Model Evaluation

In [ ]:
model.eval() # Set model to evaluation mode

with torch.no_grad():
    y_pred_tensor = model(X_test_tensor)
    test_loss = criterion(y_pred_tensor, y_test_tensor)
    print(f'Test MSE Loss: {test_loss.item():.4f}')

    y_pred = y_scaler.inverse_transform(y_pred_tensor.numpy())
    print(f"Y Predicted: {y_pred[:5].flatten()}")

In [ ]:
new_house_features = np.array([[2400, 4, 3]]) 
new_house_scaled = scaler.transform(new_house_features)
print(f"Shape of new house features: {new_house_scaled.shape}, {new_house_features.shape}")

new_house_tensor = torch.tensor(new_house_scaled, dtype=torch.float32)
print(f"New House Tensor {new_house_tensor}")

with torch.no_grad():
    predicted_price_scaled = model(new_house_tensor)  # prediction in scaled space (shape (1,1))
    print(f'Predicted Price for the new house (scaled): {predicted_price_scaled.item():.4f}')
    
    predicted_price_unscaled = y_scaler.inverse_transform(predicted_price_scaled.numpy()) # inverse transform to original price scale
    print(f"Predicted Price for the new house: ${predicted_price_unscaled[0,0]:,.2f}")

### Visualize Predictions vs. Actual

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.7)
plt.xlabel('Actual Prices')
plt.ylabel('Predicted Prices')
plt.title('Actual vs. Predicted Prices')
plt.plot([min(y_test), max(y_test)], [min(y_test), max(y_test)], color='red', linestyle='--')
plt.show()

## 8. Save, Load, and Infer

In [ ]:
MODEL_NAME = 'multivariate_regression_data_model.pth'

In [ ]:
import os
if os.path.exists(MODEL_NAME):
    os.remove(MODEL_NAME)
    print("Existing Model removed successfully.")

torch.save(model.state_dict(), MODEL_NAME)
print("Model saved successfully.")

### Load the Model and Predict a New House Price

In [ ]:
loaded_model = RegressionModel(input_size)
loaded_model.load_state_dict(torch.load(MODEL_NAME))
loaded_model.eval() # Set model to evaluation mode

## Inference

In [ ]:
new_house_features = np.array([
    [1500, 3, 2],
    [2400, 4, 3],
    [2000, 4, 3],
    [1100, 2, 1.5],
    [3000, 5, 4],
    [1800, 3, 2.5]
]) 

new_house_scaled = scaler.transform(new_house_features) # Try without scaling

# Predict for all new houses and print results
new_house_tensor_all = torch.tensor(new_house_scaled, dtype=torch.float32)
with torch.no_grad():
    preds_scaled = loaded_model(new_house_tensor_all)
    
    ## Inverse transform to original price scale
    preds_unscaled = y_scaler.inverse_transform(preds_scaled.numpy()) # Test

for feat, pred in zip(new_house_features, preds_unscaled):
    print(f"Features {feat} -> Predicted Price: ${pred[0]:,.2f}")

## Conclusion

- `Loaded house price data from CSV` and inspected structure, types, and summary statistics.
- Visualized feature relationships (`pairplot`) and feature correlations (`heatmap`) to inform modeling.
- Selected features (`SquareFeet, NumBedrooms, NumBathrooms`) and target (Price); `split into training and test sets`.
- Standardized input features and target values with `StandardScaler` for stable training.
- Converted data to PyTorch tensors and created a DataLoader for `mini-batch training`.
- Defined a feedforward neural network (RegressionModel) in PyTorch for regression.
- Trained the model for 100 epochs using Adam optimizer and MSE loss; tracked and plotted training loss.
- Evaluated the trained model on the test set (reported test MSE) and visualized Actual vs Predicted prices.
- Performed `inference on new, unseen house samples`; demonstrated `saving and loading` the model state dict.
- Saved the trained model to disk (MODEL_NAME) to enable future reuse and deployment.